# Quora Duplicate Questions — EDA & Baseline Model

## 1. Environment Setup

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import scipy.sparse as sp
from collections import Counter
from datasets import load_dataset
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

True

## 2. Data Loading

In [8]:
ds = load_dataset(
    "sentence-transformers/quora-duplicates",
    "pair-class"
)
df = ds['train'].to_pandas()
print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

Dataset loaded: 404,290 rows, 3 columns


## 3. EDA — Basic Inspection

In [9]:
df.head()

,sentence1,sentence2,label
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [10]:
df.shape

(404290, 3)

In [11]:
df.describe()

,label
count,404290.000000
mean,0.369198
std,0.482588
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,1.000000


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404290 entries, 0 to 404289
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   sentence1  404290 non-null  object
 1   sentence2  404290 non-null  object
 2   label      404290 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 9.3+ MB


## 4. Noise & Missing Value Analysis

In [13]:
print("Null values per column:")
print(df.isnull().sum())

print(f"\nDuplicate rows: {df.duplicated().sum()}")

Null values per column:
sentence1    0
sentence2    0
label        0
dtype: int64

Duplicate rows: 0


In [14]:
# Drop rows where either sentence is null
df.dropna(subset=['sentence1', 'sentence2'], inplace=True)

# Drop fully duplicate rows
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Shape after cleaning: {df.shape}")

Shape after cleaning: (404290, 3)


## 5. Label Distribution

In [15]:
df['label'].value_counts().plot(kind='bar', color=['steelblue', 'coral'])
plt.xlabel("Label (0 = Non-Duplicate, 1 = Duplicate)")
plt.ylabel("Count")
plt.title("Duplicate vs Non-Duplicate Questions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(df['label'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

label
0    63.08%
1    36.92%
Name: proportion, dtype: object


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25828\3271799806.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Text Length Analysis

In [16]:
df['que1_length'] = df['sentence1'].apply(len)
df['que2_length'] = df['sentence2'].apply(len)
df[['que1_length', 'que2_length']].describe()

,que1_length,que2_length
count,404290.000000,404290.000000
mean,59.536716,60.108365
std,29.940641,33.863870
min,1.000000,0.000000
25%,39.000000,39.000000
50%,52.000000,51.000000
75%,72.000000,72.000000
max,623.000000,1169.000000


In [17]:
plt.figure(figsize=(10, 5))
plt.hist(df['que1_length'], bins=30, alpha=0.7, label='Sentence 1')
plt.hist(df['que2_length'], bins=30, alpha=0.7, label='Sentence 2')
plt.title("Character Length Distribution")
plt.xlabel("Length")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25828\2610017929.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
df['que1_words'] = df['sentence1'].apply(lambda x: len(x.split()))
df['que2_words'] = df['sentence2'].apply(lambda x: len(x.split()))
print(df[['que1_words', 'que2_words']].describe())

          que1_words     que2_words
count  404290.000000  404290.000000
mean       10.942210      11.181986
std         5.428824       6.305254
min         1.000000       0.000000
25%         7.000000       7.000000
50%        10.000000      10.000000
75%        13.000000      13.000000
max       125.000000     237.000000


In [19]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(df['que1_words'], ax=axes[0])
axes[0].set_title("Word Count Outliers — Sentence 1")
sns.boxplot(df['que2_words'], ax=axes[1])
axes[1].set_title("Word Count Outliers — Sentence 2")
plt.tight_layout()
plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25828\2695412655.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. NLP Pipeline

### 7a. Tokenization

In [20]:
# Tokenize using NLTK word_tokenize
df['tokens_q1'] = df['sentence1'].apply(lambda x: word_tokenize(x.lower()))
df['tokens_q2'] = df['sentence2'].apply(lambda x: word_tokenize(x.lower()))

print("Sample tokens (sentence1):")
print(df['tokens_q1'][0])
print("\nSample tokens (sentence2):")
print(df['tokens_q2'][0])

Sample tokens (sentence1):
['what', 'is', 'the', 'step', 'by', 'step', 'guide', 'to', 'invest', 'in', 'share', 'market', 'in', 'india', '?']

Sample tokens (sentence2):
['what', 'is', 'the', 'step', 'by', 'step', 'guide', 'to', 'invest', 'in', 'share', 'market', '?']


### 7b. Most Common Words

In [21]:
stop_words = set(stopwords.words('english'))

# Flatten all tokens from sentence1, remove stopwords and punctuation
all_tokens = [
    token for tokens in df['tokens_q1']
    for token in tokens
    if token.isalpha() and token not in stop_words
]

common_words = Counter(all_tokens).most_common(20)
words = [i[0] for i in common_words]
counts = [i[1] for i in common_words]

plt.figure(figsize=(12, 5))
plt.bar(words, counts, color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title("Top 20 Most Common Words (sentence1, stopwords removed)")
plt.tight_layout()
plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25828\3186340227.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7c. Text Cleaning

In [22]:
def clean_text(text):
    """Lowercase and remove special characters from text."""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)
    return text

df['clean_q1'] = df['sentence1'].apply(clean_text)
df['clean_q2'] = df['sentence2'].apply(clean_text)

df[['sentence1', 'clean_q1', 'sentence2', 'clean_q2']].head()

,sentence1,clean_q1,sentence2,clean_q2
0,What is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,what is the story of kohinoor kohinoor diamond,What would happen if the Indian government sto...,what would happen if the indian government sto...
2,How can I increase the speed of my internet co...,how can i increase the speed of my internet co...,How can Internet speed be increased by hacking...,how can internet speed be increased by hacking...
3,Why am I mentally very lonely? How can I solve...,why am i mentally very lonely how can i solve it,Find the remainder when [math]23^{24}[/math] i...,find the remainder when math2324math is divide...
4,"Which one dissolve in water quikly sugar, salt...",which one dissolve in water quikly sugar salt ...,Which fish would survive in salt water?,which fish would survive in salt water


## 8. Sample Analysis — Duplicate & Non-Duplicate Questions

In [23]:
duplicate_questions = df[df['label'] == 1]
print(f"Total duplicate pairs: {len(duplicate_questions):,}")
duplicate_questions[['sentence1', 'sentence2']].head()

Total duplicate pairs: 149,263


,sentence1,sentence2
5,Astrology: I am a Capricorn Sun Cap moon and c...,"I'm a triple Capricorn (Sun, Moon and ascendan..."
7,How can I be a good geologist?,What should I do to be a great geologist?
11,How do I read and find my YouTube comments?,How can I see all my Youtube comments?
12,What can make Physics easy to learn?,How can you make physics easy to learn?
13,What was your first sexual experience like?,What was your first sexual experience?


In [24]:
non_duplicate = df[df['label'] == 0]
print(f"Total non-duplicate pairs: {len(non_duplicate):,}")
non_duplicate[['sentence1', 'sentence2']].head()

Total non-duplicate pairs: 255,027


,sentence1,sentence2
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?


## 9. Baseline Model — TF-IDF + Logistic Regression

### 9a. TF-IDF Vectorization

In [25]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2)
)

q1_tfidf = tfidf.fit_transform(df['clean_q1'])
q2_tfidf = tfidf.transform(df['clean_q2'])

X = sp.hstack([q1_tfidf, q2_tfidf])
y = df['label']

print(f"Feature matrix shape: {X.shape}")

Feature matrix shape: (404290, 10000)


### 9b. Train-Test Split

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size: {X_train.shape[0]:,} | Test size: {X_test.shape[0]:,}")

Train size: 323,432 | Test size: 80,858


### 9c. Model Training

In [27]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("Model training complete.")

Model training complete.


### 9d. Evaluation

In [28]:
y_pred = model.predict(X_test)

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("F1 Score     :", f1_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Accuracy Score: 0.7416087461970368
F1 Score     : 0.590614284314686

              precision    recall  f1-score   support

           0       0.75      0.88      0.81     51005
           1       0.71      0.50      0.59     29853

    accuracy                           0.74     80858
   macro avg       0.73      0.69      0.70     80858
weighted avg       0.74      0.74      0.73     80858



### 9e. Confusion Matrix

In [29]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Dup', 'Duplicate'],
            yticklabels=['Non-Dup', 'Duplicate'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25828\3294775657.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 9f. Sample Predictions

In [30]:
results = df.iloc[y_test.index].copy()
results['Predicted'] = y_pred
results[['clean_q1', 'clean_q2', 'label', 'Predicted']].head(10)

,clean_q1,clean_q2,label,Predicted
17222,what type of guy should i be,what type of guy should i find as a husband,0,0
223267,i just started ketogenic diet im losing weight...,i am trying to reach ketosis to lose weight wi...,0,0
367674,how can i lose 4kg weight,how do i reduce my weight,1,1
152597,why are there so many dumb questions asked on ...,why do people ask such stupid questions on quora,1,1
280836,what would happen to an astronaut who took the...,what would really happen to an astronaut who t...,1,1
49119,when will india get fgfa fifth generation figh...,is india developing any 5th generation fighter...,1,0
256604,how will drones change local law enforcement o...,with the faa receiving over 25 reports every m...,0,0
276927,can i use my friends car for uber,can i use my friends car for lyft,0,0
73028,how can i lose weight quickly in 2 weeks,what are the best ways to lose a lot of weight...,1,0
263616,how can i know if im brainwashed and how do i ...,how do i know if i am an introvert or an extro...,0,1
